<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Combined First & Second Order Markov Chain Evaluation
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:18px;font-family:Arial'><b>**Prerequisites:**</b></p> 
<p style = 'font-size:16px;font-family:Arial'>Notebooks 2 and 2.1 must have been run so that the per-class model tables exist:</p>
<li style = 'font-size:16px;font-family:Arial'>`mk_model_<target>` — first-order (Event_1, Event_2) transition probabilities</li>
<li style = 'font-size:16px;font-family:Arial'>`mk2_model_<target>` — second-order (Event_1, Event_2, Event_3) transition probabilities</li>
</p>

<p style = 'font-size:18px;font-family:Arial'><b>**What this notebook does:**</b></p>
<li style = 'font-size:16px;font-family:Arial'>1. Scores every test session against both model orders for each target class</li>
<li style = 'font-size:16px;font-family:Arial'>2. Normalizes log-odds by transitions per session (so the two scales are comparable)</li>
<li style = 'font-size:16px;font-family:Arial'>3. Combines them:  `combined = α × norm_log_odds_1st + (1-α) × norm_log_odds_2nd`</li>
<li style = 'font-size:16px;font-family:Arial'>4. Grid-searches α ∈ [0, 1] to find the best value (by weighted F1)</li>
<li style = 'font-size:16px;font-family:Arial'>5. Produces full evaluation (confusion matrix, classification report) at the best α</li></p>

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from teradataml import *
import getpass
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, f1_score
)

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=2.2_Bank_ClickStream_-_Combined_Markov_Chain_Evaluation.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>2. Configuration</b></p>


In [ ]:
# training_table = "banking_v1_train"
# test_holdout_table = "banking_v1_test"
training_table = "DEMO_Bank.Session_Events_Train"
test_holdout_table = "DEMO_Bank.Session_Events_Test"

In [ ]:
# Discover target classes (same as notebooks 2 and 2.1)
apply_events_df = DataFrame.from_query(f"""
    SELECT DISTINCT Event FROM {training_table} WHERE Event LIKE 'Apply%'
""")
apply_targets = apply_events_df.to_pandas()['Event'].tolist()
print(f"Discovered {len(apply_targets)} target classes: {apply_targets}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Verify Pre-trained Model Tables Exist</b></p>

In [ ]:
print("Checking model tables...")
for target in apply_targets:
    for prefix, order in [("mk_model_", "1st"), ("mk2_model_", "2nd")]:
        tbl = f"{prefix}{target}"
        try:
            n = DataFrame(tbl).shape[0]
            print(f"  {order}-order  {tbl}: {n} rows")
        except Exception as e:
            print(f"  ERROR: {tbl} not found! Run notebook {'2' if order == '1st' else '2.1'} first.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>4. Score Test Sessions — First-Order Models (all classes)</b></p>

In [ ]:
# First-order nPath: A.B
npath_1st = f"""
    SELECT UserId, SessionId, Event_1, Event_2
    FROM npath(
        ON (SELECT * FROM {test_holdout_table})
        PARTITION BY UserId, SessionId ORDER BY \"Event_TS\"
        USING mode(overlapping) pattern('A.B')
        Symbols(TRUE AS A, TRUE AS B)
        Result(first(UserId OF A) AS UserId, first(SessionId OF A) AS SessionId,
               first(Event OF A) AS Event_1, first(Event OF B) AS Event_2)
        Filter(FIRST(\"Event_TS\" + INTERVAL '1' DAY OF ANY(A)) > FIRST(\"Event_TS\" OF ANY(B)))
    )
"""

score_parts_1st = []
for target in apply_targets:
    mt = f"mk_model_{target}"
    score_parts_1st.append(f"""
        SELECT UserId, SessionId, CAST('{target}' AS VARCHAR(50)) AS target_class,
               COUNT(*) AS transitions,
               SUM(LOG(o.probability / n.probability)) AS log_odds
        FROM ({npath_1st}) a
        JOIN {mt} o ON o.outcome=1 AND a.Event_1=o.Event_1 AND a.Event_2=o.Event_2
        JOIN {mt} n ON n.outcome=0 AND a.Event_1=n.Event_1 AND a.Event_2=n.Event_2
        GROUP BY UserId, SessionId
    """)

union_1st = " UNION ALL ".join(score_parts_1st)

try:
    execute_sql("DROP TABLE combo_scores_1st")
except:
    pass

execute_sql(f"""
    CREATE TABLE combo_scores_1st, STORAGE = TD_OFSSTORAGE AS (
        {union_1st}
    ) WITH DATA PRIMARY INDEX(UserId, SessionId, target_class)
""")

print(f"First-order scores: {DataFrame('combo_scores_1st').shape[0]} (session, class) pairs")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>5. Score Test Sessions — Second-Order Models (all classes)</b></p>

In [ ]:
# Second-order nPath: A.B.C
npath_2nd = f"""
    SELECT UserId, SessionId, Event_1, Event_2, Event_3
    FROM npath(
        ON (SELECT * FROM {test_holdout_table})
        PARTITION BY UserId, SessionId ORDER BY \"Event_TS\"
        USING mode(overlapping) pattern('A.B.C')
        Symbols(TRUE AS A, TRUE AS B, TRUE AS C)
        Result(first(UserId OF A) AS UserId, first(SessionId OF A) AS SessionId,
               first(Event OF A) AS Event_1, first(Event OF B) AS Event_2, first(Event OF C) AS Event_3)
        Filter(FIRST(\"Event_TS\" + INTERVAL '1' DAY OF ANY(A)) > FIRST(\"Event_TS\" OF ANY(C)))
    )
"""

score_parts_2nd = []
for target in apply_targets:
    mt = f"mk2_model_{target}"
    score_parts_2nd.append(f"""
        SELECT UserId, SessionId, CAST('{target}' AS VARCHAR(50)) AS target_class,
               COUNT(*) AS transitions,
               SUM(LOG(o.probability / n.probability)) AS log_odds
        FROM ({npath_2nd}) a
        JOIN {mt} o ON o.outcome=1 AND a.Event_1=o.Event_1 AND a.Event_2=o.Event_2 AND a.Event_3=o.Event_3
        JOIN {mt} n ON n.outcome=0 AND a.Event_1=n.Event_1 AND a.Event_2=n.Event_2 AND a.Event_3=n.Event_3
        GROUP BY UserId, SessionId
    """)

union_2nd = " UNION ALL ".join(score_parts_2nd)

try:
    execute_sql("DROP TABLE combo_scores_2nd")
except:
    pass

execute_sql(f"""
    CREATE TABLE combo_scores_2nd, STORAGE = TD_OFSSTORAGE AS (
        {union_2nd}
    ) WITH DATA PRIMARY INDEX(UserId, SessionId, target_class)
""")

print(f"Second-order scores: {DataFrame('combo_scores_2nd').shape[0]} (session, class) pairs")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>6. Build Ground Truth</b></p>

In [ ]:
try:
    execute_sql("DROP TABLE combo_ground_truth")
except:
    pass

execute_sql(f"""
    CREATE TABLE combo_ground_truth, STORAGE = TD_OFSSTORAGE AS (
        SELECT UserId, SessionId,
               CAST(COALESCE(MAX(CASE WHEN Event LIKE 'Apply%' THEN Event END), 'NoApplication') AS VARCHAR(50)) AS true_class
        FROM {test_holdout_table}
        GROUP BY UserId, SessionId
    ) WITH DATA PRIMARY INDEX(UserId, SessionId)
""")

gt_df = DataFrame('combo_ground_truth').to_pandas().reset_index()
gt_df.columns = gt_df.columns.str.lower()
print("Ground truth distribution:")
print(gt_df['true_class'].value_counts())
print("gt_df columns:", gt_df.columns.tolist())  # confirm it worked

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>7. Pull Scores to Pandas & Normalize</b></p>

In [ ]:
scores_1st = DataFrame('combo_scores_1st').to_pandas().reset_index()
scores_2nd = DataFrame('combo_scores_2nd').to_pandas().reset_index()

print(f"1st-order score rows: {len(scores_1st)}")
print(f"2nd-order score rows: {len(scores_2nd)}")

scores_1st.columns = scores_1st.columns.str.lower()
scores_2nd.columns = scores_2nd.columns.str.lower()

In [ ]:
scores_1st

In [ ]:
# Normalize log-odds by number of transitions per session (makes scales comparable)
scores_1st['norm_log_odds'] = scores_1st['log_odds'] / scores_1st['transitions']
scores_2nd['norm_log_odds'] = scores_2nd['log_odds'] / scores_2nd['transitions']

In [ ]:
# Merge the two score sets on (UserId, SessionId, target_class)
merged = scores_1st[['userid', 'sessionid', 'target_class', 'norm_log_odds']].rename(
    columns={'norm_log_odds': 'nlo_1st'}
).merge(
    scores_2nd[['userid', 'sessionid', 'target_class', 'norm_log_odds']].rename(
        columns={'norm_log_odds': 'nlo_2nd'}
    ),
    on=['userid', 'sessionid', 'target_class'],
    how='outer'
)

# Fill NaN for sessions that only have one order's score (e.g., too few events for 2nd order)
merged['nlo_1st'] = merged['nlo_1st'].fillna(0)
merged['nlo_2nd'] = merged['nlo_2nd'].fillna(0)

print(f"Merged score rows: {len(merged)}")
merged.head(10)

In [ ]:
# Quick look at scale differences
print("1st-order normalized log-odds stats:")
print(merged['nlo_1st'].describe())
print("\n2nd-order normalized log-odds stats:")
print(merged['nlo_2nd'].describe())

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>8. Grid Search over α</b></p>
<p style = 'font-size:16px;font-family:Arial'>For each α, combine scores → pick best class per session → compare to ground truth → record metrics.</p>
<p style = 'font-size:16px;font-family:Arial'>Also evaluates α=1.0 (pure first-order) and α=0.0 (pure second-order) as baselines.</p>

In [ ]:
def evaluate_alpha(merged_df, gt_df, alpha):
    """
    Given merged scores and ground truth, combine with weight alpha,
    classify each session, and return metrics.
    """
    df = merged_df.copy()
    df['combined'] = alpha * df['nlo_1st'] + (1 - alpha) * df['nlo_2nd']
    
    # For each session, pick the class with the highest combined score
    best = df.loc[df.groupby(['userid', 'sessionid'])['combined'].idxmax()]
    best = best[['userid', 'sessionid', 'target_class', 'combined']].copy()
    best.columns = ['userid', 'sessionid', 'predicted_class', 'combined']
    
    # If best combined < 0, classify as NoApplication
    best.loc[best['combined'] < 0, 'predicted_class'] = 'NoApplication'
    
    # Join with ground truth
    eval_df = gt_df.merge(best[['userid', 'sessionid', 'predicted_class']],
                          on=['userid', 'sessionid'], how='left')
    eval_df['predicted_class'] = eval_df['predicted_class'].fillna('NoApplication')
    
    acc = accuracy_score(eval_df['true_class'], eval_df['predicted_class'])
    wf1 = f1_score(eval_df['true_class'], eval_df['predicted_class'], average='weighted', zero_division=0)
    mf1 = f1_score(eval_df['true_class'], eval_df['predicted_class'], average='macro', zero_division=0)
    
    return acc, wf1, mf1, eval_df

In [ ]:
merged

In [ ]:
# Grid search
alphas = np.arange(0.0, 1.05, 0.05)
results = []

for alpha in alphas:
    acc, wf1, mf1, _ = evaluate_alpha(merged, gt_df, alpha)
    results.append({'alpha': round(alpha, 2), 'accuracy': acc, 'weighted_f1': wf1, 'macro_f1': mf1})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Plot alpha vs metrics
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(results_df['alpha'], results_df['accuracy'], 'o-', label='Accuracy', linewidth=2)
ax.plot(results_df['alpha'], results_df['weighted_f1'], 's-', label='Weighted F1', linewidth=2)
ax.plot(results_df['alpha'], results_df['macro_f1'], '^-', label='Macro F1', linewidth=2)

# Mark the best weighted F1
best_row = results_df.loc[results_df['weighted_f1'].idxmax()]
ax.axvline(best_row['alpha'], color='red', linestyle='--', alpha=0.7, label=f"Best α = {best_row['alpha']:.2f}")

ax.set_xlabel('α  (1.0 = pure 1st-order, 0.0 = pure 2nd-order)')
ax.set_ylabel('Score')
ax.set_title('Grid Search: Combined Markov Model — α vs Evaluation Metrics')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Identify best alpha by weighted F1
best_alpha = results_df.loc[results_df['weighted_f1'].idxmax(), 'alpha']
print(f"Best α = {best_alpha:.2f}")
print(f"  Accuracy:    {results_df.loc[results_df['weighted_f1'].idxmax(), 'accuracy']:.4f}")
print(f"  Weighted F1: {results_df.loc[results_df['weighted_f1'].idxmax(), 'weighted_f1']:.4f}")
print(f"  Macro F1:    {results_df.loc[results_df['weighted_f1'].idxmax(), 'macro_f1']:.4f}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>9. Full Evaluation at Best α</b></p>

In [ ]:
_, _, _, eval_best = evaluate_alpha(merged, gt_df, best_alpha)

print(f"Total test sessions: {len(eval_best)}")
print(f"\nOverall Accuracy: {accuracy_score(eval_best['true_class'], eval_best['predicted_class']):.4f}")
print(f"\nClassification Report (α = {best_alpha:.2f}):")
print(classification_report(eval_best['true_class'], eval_best['predicted_class'], zero_division=0))

In [ ]:
# Confusion matrix at best alpha
labels = sorted(eval_best['true_class'].unique())
cm = confusion_matrix(eval_best['true_class'], eval_best['predicted_class'], labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Counts) — Combined α={best_alpha:.2f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title(f'Confusion Matrix (Normalized) — Combined α={best_alpha:.2f}')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class summary table
p, r, f1, sup = precision_recall_fscore_support(
    eval_best['true_class'], eval_best['predicted_class'], labels=labels, zero_division=0
)
summary = pd.DataFrame({
    'Class': labels, 'Precision': p, 'Recall': r, 'F1': f1, 'Support': sup
}).set_index('Class')
print(summary.to_string())
print(f"\nWeighted F1: {(summary['F1'] * summary['Support']).sum() / summary['Support'].sum():.4f}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>10. Compare: Pure 1st-Order vs Pure 2nd-Order vs Best Combined</b></p>

In [ ]:
comparison = []
for label, a in [('Pure 1st-order (α=1.0)', 1.0), ('Pure 2nd-order (α=0.0)', 0.0), (f'Best Combined (α={best_alpha:.2f})', best_alpha)]:
    acc, wf1, mf1, _ = evaluate_alpha(merged, gt_df, a)
    comparison.append({'Model': label, 'Accuracy': f'{acc:.4f}', 'Weighted F1': f'{wf1:.4f}', 'Macro F1': f'{mf1:.4f}'})

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

In [ ]:
# Side-by-side confusion matrices for the three configurations
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, (label, a) in zip(axes, [('1st-Order Only', 1.0), ('2nd-Order Only', 0.0), (f'Combined α={best_alpha:.2f}', best_alpha)]):
    _, _, _, ev = evaluate_alpha(merged, gt_df, a)
    cm_i = confusion_matrix(ev['true_class'], ev['predicted_class'], labels=labels)
    cm_n = cm_i.astype(float) / cm_i.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>11. Cleanup</b></p>

In [ ]:
# Discover all Apply* events from training data
training_table = "DEMO_Bank.Session_Events_Train"
apply_events_df = DataFrame.from_query("""
    SELECT DISTINCT Event FROM {0} WHERE Event LIKE 'Apply%'
""".format(training_table))
apply_targets = apply_events_df.to_pandas()['Event'].tolist()
print(f"Discovered {len(apply_targets)} target classes: {apply_targets}")

In [ ]:
for tbl in ['combo_scores_1st', 'combo_scores_2nd', 'combo_ground_truth']:
    try: db_drop_table(tbl)
    except: pass
# Cleanup multiclass tables (optional)
for target in apply_targets:
    try: db_drop_table(f"mk2_model_{target}")
    except: pass
for tbl in ['mc2_nb_all_scores', 'mc2_predictions', 'mc2_ground_truth']:
    try: db_drop_table(tbl)
    except: pass
# Cleanup multiclass tables (optional)
for target in apply_targets:
    try: db_drop_table(f"mk2_model_{target}")
    except: pass
for tbl in ['mc2_nb_all_scores', 'mc2_predictions', 'mc2_ground_truth']:
    try: db_drop_table(tbl)
    except: pass

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>